In [ ]:
import spiceypy as spice
import os 
import numpy as np
from scipy.integrate import solve_ivp
from functions.two_body_problem import two_body_problem

cur_dir = os.path.dirname(os.path.realpath(__name__))
spice.furnsh(os.path.join(cur_dir, 'spice_kernels', 'naif0012.tls'))
spice.furnsh(os.path.join(cur_dir, 'spice_kernels', 'gm_de440.tpc'))
spice.furnsh(os.path.join(cur_dir, 'spice_kernels', 'pck00011.tpc'))
spice.furnsh(os.path.join(cur_dir, 'spice_kernels', 'de442.bsp'))
 

In [ ]:
t0_utc_et = spice.str2et("2025-09-24T00:00:00")
t0_utc_str = spice.et2utc(t0_utc_et, "C", 0)
print(f"Epoch (UTC): {t0_utc_str}")

mu_sun = spice.bodvrd("SUN","GM",3)[1][0]   
print(f"Gravitational parameter of the Sun: {mu_sun} km^3/s^2")

rv0_earth = spice.spkezr("EARTH",t0_utc_et,"ECLIPJ2000","NONE","SUN")
print(rv0_earth)
r0_earth = rv0_earth[0][0:3]
print(f"Position vector of Earth relative to Sun at epoch: {r0_earth}")
v0_earth = rv0_earth[0][3:6]
print(f"Velocity vector of Earth relative to Sun at epoch: {v0_earth}")

kep_earth = spice.oscltx(rv0_earth[0], t0_utc_et, mu_sun)
print(f"Keplerian elements of Earth's orbit at epoch: {kep_earth}")

a_earth = kep_earth[9]
t_earth = 2*(22/7)*((a_earth**3)/mu_sun)**0.5
print(f"Orbital period of Earth: {t_earth/(60*60*24)} days")

t_span = [0,t_earth]
t_eval = np.linspace(0, t_earth, 100)
rv_i_earth = solve_ivp(two_body_problem, t_span=t_span, y0=np.concatenate([r0_earth, v0_earth]), method='DOP853', atol=1e-10, rtol=1e-10,args=(mu_sun,),t_eval=t_eval)


